# Power BI Copilot Diagnostics — Parser & Troubleshooter

This notebook parses a **Power BI Copilot diagnostic** file (`copilot_diagnostic_data_*.txt`, a JSON document) and turns it into an actionable troubleshooting report. It highlights the signals that tell you **what to adjust in your semantic model** so Copilot answers correctly.

It surfaces:

1. **Session metadata** — who/what/when, agent, service version.
2. **Conversation turns** — each user question, the tool Copilot chose, and whether it succeeded.
3. **Per-question diagnostics** — warnings (e.g. `DataIndexNotReady`, `QueryAggregateNotSupported`, `AgentSchemaReduced`), NL‑to‑DAX **fallback reasons**, generated **DAX**, empty results, and clarifications.
4. **Model AI settings** — the AI data schema (visible vs. hidden fields), AI instructions, example prompts, and indexing state.
5. **An automated recommendations report** mapping each detected issue to the fix, following Microsoft's *Prep data for AI* guidance.

**Reference docs**
- [FAQ: Preparing Data for AI (Power BI)](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq)
- [Tutorial: Prepare Semantic Model for AI](https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model)

> Tip: In Copilot, download the diagnostics from the response's *…* menu. Warnings like `AgentSchemaReduced` only appear in these files.

## 1. Load the diagnostic file

By default this finds the most recent `copilot_diagnostic_data_*.txt` next to the notebook. Set `DIAG_PATH` explicitly to point at a specific file.

In [ ]:
from datetime import datetime
import glob
import os
import json

# DIAG_PATH may be: a specific .txt file, a FOLDER (newest matching file is picked),
# or None (search this notebook's folder). Use a raw string (r"...") for Windows paths.
DIAG_PATH = None
# Example: DIAG_PATH = r"C:\path\to\copilot_diagnostic_data_...txt"

PATTERN = 'copilot_diagnostic_data_*.txt'

def _newest_in(folder):
    matches = sorted(
        glob.glob(os.path.join(folder, PATTERN)),
        key=os.path.getmtime,
        reverse=True,
    )
    return matches[0] if matches else None

if DIAG_PATH is None:
    DIAG_PATH = _newest_in(os.getcwd())
elif os.path.isdir(DIAG_PATH):
    # A folder was given — pick the most recent diagnostic file inside it.
    DIAG_PATH = _newest_in(DIAG_PATH)

if not DIAG_PATH or not os.path.isfile(DIAG_PATH):
    raise FileNotFoundError(
        f'No {PATTERN} file found. Set DIAG_PATH to a specific diagnostic .txt file '
        'or to the folder that contains one.'
    )

with open(DIAG_PATH, 'r', encoding='utf-8') as f:
    diag = json.load(f)

print(f'Loaded: {DIAG_PATH}')
print(f'Top-level keys: {list(diag.keys())}')

## 2. Session metadata

In [ ]:
meta = {
    'Session ID': diag.get('sessionId'),
    'Copilot session ID': diag.get('copilotSessionId'),
    'Conversation ID': diag.get('conversationId'),
    'Consumption method': diag.get('consumptionMethod'),
    'Copilot agent': diag.get('CopilotAgentId'),
    'Service version': diag.get('serviceVersion'),
    'Timestamp': diag.get('timestamp'),
}

width = max(len(k) for k in meta)
for k, v in meta.items():
    print(f'{k:<{width}} : {v}')

## 3. Conversation turns

Walk `chatHistory` and pair each **user** question with the **tool** Copilot invoked and the outcome. `answerDataQuestion` = a data/Q&A question; `getDatasetSchema` = a schema/authoring request.

In [ ]:
def text_of(content):
    """Flatten the various content shapes used in chatHistory to plain text."""
    if isinstance(content, str):
        return content
    parts = []
    if isinstance(content, list):
        for item in content:
            if not isinstance(item, dict):
                continue
            if item.get('type') == 'text':
                parts.append(item.get('text', {}).get('value', ''))
    return ' '.join(p for p in parts if p).strip()

turns = []
history = diag.get('chatHistory', [])
for i, msg in enumerate(history):
    role = msg.get('role')
    if role == 'user':
        turns.append({
            'turn': len(turns) + 1,
            'question': text_of(msg.get('content')),
            'tools': [],
            'status': None,
            'answer': None,
        })
    elif role == 'assistant' and msg.get('tool_calls') and turns:
        for tc in msg['tool_calls']:
            turns[-1]['tools'].append(tc.get('function', {}).get('name'))
    elif role == 'tool' and turns:
        result = msg.get('result', {})
        turns[-1]['status'] = result.get('status', {}).get('kind')
        ans = text_of(msg.get('content'))
        if ans:
            turns[-1]['answer'] = ans

for t in turns:
    tools = ', '.join(x for x in t['tools'] if x) or '—'
    print(f"Turn {t['turn']}: {t['question']}")
    print(f"   tool(s): {tools}   status: {t['status']}")
    if t['answer']:
        preview = t['answer'][:220] + ('…' if len(t['answer']) > 220 else '')
        print(f"   answer : {preview}")
    print()

## 4. Per-question diagnostics

The `dataQuestion` object holds the deep signals for each Q&A attempt: interpretation **warnings**, whether Copilot fell back to **NL‑to‑DAX** (and why), the generated **DAX**, whether the result was **empty**, and any **clarification** it returned to the user.

In [ ]:
def deep_get(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict):
            d = d.get(k)
        elif isinstance(d, list) and isinstance(k, int) and -len(d) <= k < len(d):
            d = d[k]
        else:
            return default
    return d if d is not None else default

# Clarifications for a turn are recorded in *later* turns' contextEvents (as history),
# keyed by the utterance they belong to. Build a global map utterance -> clarificationKind.
clarification_by_utterance = {}
for q in (diag.get('dataQuestion') or {}).values():
    for ev in deep_get(q, 'interpretRequest', 'conversationalContext', 'contextEvents', default=[]):
        ck = deep_get(ev, 'responses', 0, 'command', 'clarification', 'clarificationKind')
        if ck and ev.get('utterance'):
            clarification_by_utterance[ev['utterance']] = ck

questions = []
for qid, q in (diag.get('dataQuestion') or {}).items():
    resp = q.get('interpretResponse', {})
    nl = q.get('nlToDaxDetails', {})
    activity = nl.get('activityDetails', {})

    clar_kinds = []
    ck = clarification_by_utterance.get(q.get('utterance'))
    if ck:
        clar_kinds.append(ck)

    dax_list = nl.get('daxGeneration', []) or []
    dax_query = next((d.get('daxQuery') for d in dax_list if isinstance(d, dict) and d.get('daxQuery')), None)

    questions.append({
        'id': qid,
        'utterance': q.get('utterance'),
        'warnings': resp.get('warnings', []) or [],
        'errors': resp.get('errors', []) or [],
        'restatements': [r for r in (resp.get('restatements') or []) if r],
        'fallback_reason': activity.get('fallbackReason'),
        'empty_result': activity.get('emptyResult'),
        'truncated_result': activity.get('truncatedResult'),
        'dax': dax_query,
        'answer': deep_get(nl, 'textualAnswer', 'answer'),
        'context_clarifications': clar_kinds,
    })

for q in questions:
    print('=' * 90)
    print(f"Q: {q['utterance']}")
    print(f"   warnings         : {q['warnings'] or '—'}")
    print(f"   errors           : {q['errors'] or '—'}")
    if q['restatements']:
        print(f"   restated as      : {q['restatements']}")
    if q['fallback_reason']:
        print(f"   NL->DAX fallback : {q['fallback_reason']}")
    if q['empty_result']:
        print(f"   empty result     : {q['empty_result']}")
    if q['context_clarifications']:
        print(f"   clarification    : {q['context_clarifications']}")
    if q['answer']:
        print(f"   answer           : {q['answer']}")
    if q['dax']:
        print('   generated DAX    :')
        for line in q['dax'].splitlines():
            print(f'       {line}')
    print()

## 5. Troubleshooting knowledge base

Each known signal (warning, fallback reason, or clarification) maps to what it means and **what to adjust in the model**, per the *Prep data for AI* guidance.

In [ ]:
FAQ = 'https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq'
TUT = 'https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model'

# Internal Microsoft supportability project (corp auth required). Findings below add a
# deep search link so colleagues can look up known issues / remediation playbooks per signal.

KB = {
    # --- Interpretation warnings ---
    'DataIndexNotReady': {
        'meaning': "The semantic model's AI index isn't ready (model is indexing/syncing). Copilot may miss instance values (e.g. specific product or region names) until indexing completes.",
        'fix': "Wait for indexing to finish, then retry. Ensure Q&A is enabled (IndexingEnabled) so instance values are indexed. Republish or refresh to trigger reindexing. The first index can take ~15 extra minutes; import models reindex on refresh, DirectQuery/Direct Lake at most once per 24h.",
        'link': FAQ,
    },
    'QueryAggregateNotSupported': {
        'meaning': 'Q&A could not perform the requested aggregation/calculation over the fields as asked, so it could not answer natively (often triggering an NL-to-DAX fallback or a partial answer).',
        'fix': 'Add an explicit measure for the calculation the user wants, then add an AI instruction mapping the phrasing to that measure. For common asks, set a Verified Answer on a visual that already computes it.',
        'link': FAQ,
    },
    'AgentSchemaReduced': {
        'meaning': 'The model schema was too large for Copilot to process, so it automatically reduced the schema. Fields you expect may be dropped from consideration.',
        'fix': 'Simplify the AI data schema: hide tables/columns/measures that end users never query. Keep within limits (~1,000 model entities and 5M instance values). Remove duplicate/similar field names across tables.',
        'link': FAQ,
    },
    'DataIndexLimitReached': {
        'meaning': 'Indexing hit the upper bound (5M instance values). Copilot answers from a partial index, so some instance values are not searchable.',
        'fix': 'Hide high-cardinality text columns that are not needed for questions from the AI schema, and avoid indexing very large lookup columns. Split or reduce the model so key columns index first (smallest cardinality is indexed first).',
        'link': FAQ,
    },
    # --- NL-to-DAX fallback reasons ---
    'emptyInterpretation': {
        'meaning': 'Q&A could not interpret the question against the model, so Copilot fell back to generating DAX directly. This usually means naming/terminology or schema gaps.',
        'fix': 'Improve field/table names (human-readable, unique), add descriptions, and add synonyms/Terms for the words users actually use. Add AI instructions for business terms and define verified answers for common questions.',
        'link': TUT,
    },
    # --- Clarification kinds ---
    'AdvancedQueryLimitation': {
        'meaning': 'The user asked for something the model cannot do (e.g. forecasting/prediction of future values, or an advanced calculation not present in the model).',
        'fix': 'If the capability is expected, add it to the model (e.g. a forecast measure or calculation). Otherwise set user expectations via AI instructions describing what the model can/cannot answer.',
        'link': FAQ,
    },
}

# Signals that are not a named warning but still worth flagging.
EMPTY_RESULT_GUIDANCE = {
    'meaning': 'The generated DAX ran but returned a blank/zero result. Either the data genuinely has no rows for that filter, or a field/term was mapped incorrectly (wrong table, relationship, or value).',
    'fix': 'Confirm the data exists for that filter. If mapping is wrong, add AI instructions to map the term to the correct field/measure, verify relationships (star schema), and ensure the value is indexed (Q&A enabled).',
    'link': TUT,
}

print(f'Knowledge base covers {len(KB)} named signals + empty-result heuristic.')

## 6. Model AI settings (Prep data for AI)

`copilotModelSettings` is the model's AI configuration: the **AI data schema** (each field's `Visibility` — `1` visible to Copilot, `0` hidden), **AI instructions** (`CustomInstructions`), **example prompts**, and whether **indexing** is on.

In [ ]:
settings = diag.get('copilotModelSettings', {}) or {}
entities = settings.get('Entities', []) or []

tables_visible, tables_hidden = set(), set()
cols_visible, cols_hidden = [], []
terms_defined = []

for e in entities:
    b = e.get('Binding', {})
    ent = b.get('EntityName')
    prop = b.get('PropertyName')
    vis = e.get('Visibility')
    terms = e.get('Terms') or []
    if terms:
        terms_defined.append((f'{ent}[{prop}]' if prop else ent, terms))
    if prop is None:
        (tables_visible if vis == 1 else tables_hidden).add(ent)
    else:
        (cols_visible if vis == 1 else cols_hidden).append(f'{ent}[{prop}]')

print(f'Entities in AI schema     : {len(entities)}')
print(f'Tables visible to Copilot : {len(tables_visible)}')
print(f'Tables hidden from Copilot: {len(tables_hidden)}')
print(f'Columns/measures visible  : {len(cols_visible)}')
print(f'Columns/measures hidden   : {len(cols_hidden)}')
print(f'Fields with synonyms/Terms: {len(terms_defined)}')
print()
print('Indexing enabled          :', deep_get(settings, 'Settings', 'IndexingEnabled'))
print()
print('Visible tables:', sorted(tables_visible) or '—')
print()
print('Example prompts:')
for p in deep_get(settings, 'ExamplePrompts', 'Prompts', default=[]):
    print(f'  - {p}')
print()
print('AI instructions (CustomInstructions):')
ci = settings.get('CustomInstructions') or ''
print('  ' + (ci.strip().replace('\n', '\n  ') if ci.strip() else '(none set)'))

## 7. Automated troubleshooting report

Combines everything above into a prioritized list of findings and recommended model adjustments.

In [ ]:
findings = []

def add_finding(severity, signal, where, kb_entry):
    findings.append({
        'severity': severity,
        'signal': signal,
        'where': where,
        'meaning': kb_entry['meaning'],
        'fix': kb_entry['fix'],
        'link': kb_entry['link'],
    })

for q in questions:
    label = q['utterance'] or q['id']
    for e in q['errors']:
        add_finding('ERROR', f'Error: {e}', label, {
            'meaning': 'The Copilot Q&A engine reported an error while answering this question.',
            'fix': 'Review the error detail. Common causes: the model could not interpret the request, a DAX/query error, or a transient service issue. Retry after indexing completes; if it persists, simplify the question, fix model relationships/measures, or check Power BI service health.',
            'link': FAQ,
        })
    for w in q['warnings']:
        kb = KB.get(w, {
            'meaning': 'Unrecognized warning. Review the diagnostic context.',
            'fix': 'Check the Prep data for AI FAQ for this warning.',
            'link': FAQ,
        })
        add_finding('WARNING', w, label, kb)
    if q['fallback_reason'] and q['fallback_reason'] in KB:
        add_finding('INFO', f"NL->DAX fallback ({q['fallback_reason']})", label, KB[q['fallback_reason']])
    for ck in q['context_clarifications']:
        if ck in KB:
            add_finding('INFO', f'Clarification ({ck})', label, KB[ck])
    if q['empty_result'] or (q['answer'] and 'no ' in (q['answer'] or '').lower() and 'sold' in (q['answer'] or '').lower()):
        add_finding('CHECK', 'Empty/zero result', label, EMPTY_RESULT_GUIDANCE)

# Failed conversation turns (tool status other than Succeeded)
for t in turns:
    status = t.get('status')
    if status and status != 'Succeeded':
        add_finding('ERROR', f'Tool call {status}', t['question'], {
            'meaning': f"Copilot's tool call for this question did not succeed (status: {status}).",
            'fix': 'Inspect the turn in section 3. Retry after indexing; if it keeps failing, review the model (relationships/measures), rephrase the question, and check Power BI service health.',
            'link': FAQ,
        })

# Model-level checks
if deep_get(settings, 'Settings', 'IndexingEnabled') is False:
    add_finding('WARNING', 'Indexing disabled', 'Model settings', {
        'meaning': 'Q&A indexing is off, so Copilot cannot search instance values in the model.',
        'fix': 'Enable the Q&A setting on the semantic model so Power BI indexes text columns.',
        'link': FAQ,
    })

if len(entities) > 900:
    add_finding('CHECK', f'Large AI schema ({len(entities)} entities)', 'Model settings', KB['AgentSchemaReduced'])

if not (settings.get('CustomInstructions') or '').strip():
    add_finding('CHECK', 'No AI instructions', 'Model settings', {
        'meaning': 'No AI instructions are set. Copilot has no business-term or mapping context beyond names/descriptions.',
        'fix': 'Add AI instructions to define business terms, map user phrasing to fields/measures, and guide interpretation.',
        'link': TUT,
    })

order = {'ERROR': 0, 'WARNING': 1, 'CHECK': 2, 'INFO': 3}
findings.sort(key=lambda f: order.get(f['severity'], 4))

if not findings:
    print('No issues detected in this diagnostic file.')
else:
    n_err = sum(1 for f in findings if f['severity'] == 'ERROR')
    n_warn = sum(1 for f in findings if f['severity'] == 'WARNING')
    print(f'{len(findings)} finding(s)  ({n_err} error, {n_warn} warning):\n')
    for i, f in enumerate(findings, 1):
        print(f"[{f['severity']}] {i}. {f['signal']}  —  ({f['where']})")
        print(f"    What it means : {f['meaning']}")
        print(f"    What to adjust: {f['fix']}")
        print(f"    Reference     : {f['link']}")
        print()

## 8. What to adjust — checklist

Apply *Prep data for AI* features in this recommended order ([FAQ](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq)):

1. **Simplify the AI data schema** — hide tables/columns/measures users never query; remove duplicate or similar field names; keep within limits (~1,000 entities, 5M instance values). Prevents `AgentSchemaReduced` and wrong-field selection.
2. **Create verified answers** — pin trusted visuals to common/nuanced questions via trigger phrases. Best fix for frequent asks and for `QueryAggregateNotSupported`.
3. **Add AI instructions** — define business terms, seasons, synonyms, and term→field mappings. Best fix for `emptyInterpretation` fallbacks and terminology mismatches.
4. **Add descriptions** — table/column/measure descriptions (used by search and DAX queries). Improves discovery.
5. **Confirm indexing / Q&A enabled** — required for instance-value search; resolves `DataIndexNotReady`. Allow time for the first index (~15 min) and reindex after refresh/republish.
6. **Mark 'Approved for Copilot'** — after validating, signal the model is AI-ready.

**Also from the tutorial** ([link](https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model)):
- Use human-readable, unique names; follow a **star schema**; remove unused objects and ambiguous relationships.
- Add clear model, table, and column **descriptions**; add synonyms (Terms) for user vocabulary.
- **Iteratively test** with Copilot after each change and validate the answers.

> Note: For empty/zero results (e.g. "no bikes sold in 2007" while later years return values), first confirm the data actually exists for that filter; if it does, the field/term mapping or relationships likely need correcting via AI instructions and model fixes.